# Fed, Pieniądz i Rynki - Baza SQLite + Zapytania Analityczne

**Dzień 2:** Ładowanie danych do SQLite i 10 zapytań analitycznych  
**Baza:** `data/fed_cycles.db`

### Struktura bazy
- `raw_series` - wszystkie serie w formacie długim (date, series_id, value)
- `v_master` - widok szeroki: każda seria jako osobna kolumna, oś czasu miesięczna
- `dim_recession` - tabela z nazwami i datami recesji

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')
DB_PATH = Path('../data/fed_cycles.db')

# Usuń starą bazę jeśli istnieje (czysta przebudowa)
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
print(f'Baza utworzona: {DB_PATH}')

Baza utworzona: ../data/fed_cycles.db


## 1. Ładowanie CSV do tabeli `raw_series`

In [2]:
SERIES_META = {
    'FEDFUNDS': 'Stopa procentowa Fed (%)',
    'M2SL':     'Podaż pieniądza M2 (mld USD)',
    'SP500':    'Indeks S&P500 (Yahoo Finance, pelna historia od 1990)',
    'T10Y2Y':   'Spread 10Y-2Y - yield curve (pp)',
    'USREC':    'Recesja NBER (1=recesja, 0=brak)',
    'UNRATE':   'Stopa bezrobocia (%)',
    'GDPC1':    'PKB realne USA (mld USD)',
    'CPIAUCSL': 'Inflacja CPI (indeks)',
    # Seria dodane w Dniu 5 - wczesniej pobierane poza notebookiem, przez co
    # projekt nie odtwarzal sie od zera: v_master deklarowal te kolumny, a zadna
    # komorka ich nie ladowala.
    'ICSA':         'Wnioski o zasilek dla bezrobotnych (tyg.)',
    'PAYEMS':       'Zatrudnienie poza rolnictwem (tys. osob)',
    'INDPRO':       'Produkcja przemyslowa (indeks)',
    'RSXFS':        'Sprzedaz detaliczna (mln USD)',
    'PERMIT':       'Pozwolenia budowlane (tys.)',
    'BAMLH0A0HYM2': 'Spread obligacji High Yield (pp)',
    'DRTSCILM':     'Zaostrzenie warunkow kredytowych (%, kwartalnie)',
    'USSLIND':      'Wskaznik wyprzedzajacy LEI (wycofany przez FRED w 2020)',
    # Serie z Yahoo Finance, pobierane w notebooku 01: FRED bez klucza oddaje
    # SP500 tylko 10 lat, a VIX-a nie ma wcale. Wczesniej obie wchodzily do bazy
    # z notebookow 03 i 05, wiec wynik analizy zalezal od kolejnosci uruchomienia.
    'VIX':          'Indeks zmiennosci VIX (Yahoo Finance)',
}

wszystkie = []

for series_id in SERIES_META:
    sciezka = RAW_DIR / f'{series_id}.csv'
    df = pd.read_csv(sciezka, parse_dates=['date'])
    df = df[['date', 'value']].copy()
    df['series_id'] = series_id
    df = df.dropna(subset=['value'])
    wszystkie.append(df)
    print(f'{series_id:<12} {len(df):>5} wierszy')

raw = pd.concat(wszystkie, ignore_index=True)
raw['date'] = raw['date'].dt.strftime('%Y-%m-%d')
raw.to_sql('raw_series', conn, if_exists='replace', index=False)

conn.execute('CREATE INDEX IF NOT EXISTS idx_raw_date ON raw_series(date)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_raw_series ON raw_series(series_id)')
conn.commit()

print(f'\nŁącznie wierszy w raw_series: {len(raw)}')

FEDFUNDS       440 wierszy
M2SL           439 wierszy
SP500          441 wierszy
T10Y2Y        9180 wierszy
USREC          440 wierszy
UNRATE         439 wierszy
GDPC1          146 wierszy
CPIAUCSL       439 wierszy
ICSA          1914 wierszy
PAYEMS         440 wierszy
INDPRO         439 wierszy
RSXFS          415 wierszy
PERMIT         439 wierszy
BAMLH0A0HYM2  7755 wierszy
DRTSCILM       146 wierszy
USSLIND        362 wierszy
VIX            441 wierszy

Łącznie wierszy w raw_series: 24315


## 2. Tabela `dim_recession` - nazwy i daty recesji

In [3]:
# Daty NBER: start = miesiac szczytu, end = miesiac dolka.
# Recesja 1990-1991 byla wczesniej pominieta, mimo ze siedzi w serii USREC
# (8 miesiecy, 1990-08 do 1991-03). Skutek: Q4, Q7 i Q9 liczyly na 3 z 4 recesji,
# a notebook 05 dorabial jej nazwe w slowniku Pythona.
recesje = pd.DataFrame([
    {'recession_id': 1, 'name': 'Gulf War', 'start': '1990-07-01', 'end': '1991-03-01', 'cause': 'Szok naftowy po inwazji na Kuwejt i zacieśnienie Fed'},
    {'recession_id': 2, 'name': 'Dot-com',  'start': '2001-03-01', 'end': '2001-11-01', 'cause': 'Pęknięcie bańki technologicznej'},
    {'recession_id': 3, 'name': 'GFC',      'start': '2007-12-01', 'end': '2009-06-01', 'cause': 'Kryzys finansowy - rynek nieruchomości'},
    {'recession_id': 4, 'name': 'COVID',    'start': '2020-02-01', 'end': '2020-04-01', 'cause': 'Pandemia COVID-19'},
])

recesje.to_sql('dim_recession', conn, if_exists='replace', index=False)
conn.commit()
print('dim_recession zapisana:')
print(recesje.to_string(index=False))

dim_recession zapisana:
 recession_id     name      start        end                                                cause
            1 Gulf War 1990-07-01 1991-03-01 Szok naftowy po inwazji na Kuwejt i zacieśnienie Fed
            2  Dot-com 2001-03-01 2001-11-01                      Pęknięcie bańki technologicznej
            3      GFC 2007-12-01 2009-06-01               Kryzys finansowy - rynek nieruchomości
            4    COVID 2020-02-01 2020-04-01                                    Pandemia COVID-19


## 3. Widok `v_master` - szeroka tabela miesięczna

Serie dzienne (SP500, T10Y2Y) agregujemy do średniej miesięcznej żeby wszystko było na jednej osi czasu.

In [4]:
conn.execute('DROP VIEW IF EXISTS v_master')

conn.execute("""
CREATE VIEW v_master AS
WITH monthly AS (
    SELECT
        strftime('%Y-%m', date) AS ym,
        series_id,
        AVG(value) AS value
    FROM raw_series
    GROUP BY ym, series_id
)
SELECT
    ym,
    MAX(CASE WHEN series_id = 'FEDFUNDS'       THEN value END) AS fedfunds,
    MAX(CASE WHEN series_id = 'M2SL'           THEN value END) AS m2sl,
    MAX(CASE WHEN series_id = 'SP500'          THEN value END) AS sp500,
    MAX(CASE WHEN series_id = 'T10Y2Y'        THEN value END) AS t10y2y,
    MAX(CASE WHEN series_id = 'USREC'         THEN value END) AS usrec,
    MAX(CASE WHEN series_id = 'UNRATE'        THEN value END) AS unrate,
    MAX(CASE WHEN series_id = 'GDPC1'         THEN value END) AS gdpc1,
    MAX(CASE WHEN series_id = 'CPIAUCSL'      THEN value END) AS cpiaucsl,
    MAX(CASE WHEN series_id = 'ICSA'          THEN value END) AS icsa,
    MAX(CASE WHEN series_id = 'PAYEMS'        THEN value END) AS payems,
    MAX(CASE WHEN series_id = 'INDPRO'        THEN value END) AS indpro,
    MAX(CASE WHEN series_id = 'RSXFS'         THEN value END) AS rsxfs,
    MAX(CASE WHEN series_id = 'PERMIT'        THEN value END) AS permit,
    MAX(CASE WHEN series_id = 'BAMLH0A0HYM2'  THEN value END) AS hy_spread,
    MAX(CASE WHEN series_id = 'DRTSCILM'      THEN value END) AS credit_cond,
    MAX(CASE WHEN series_id = 'USSLIND'       THEN value END) AS lei,
    MAX(CASE WHEN series_id = 'VIX'           THEN value END) AS vix
FROM monthly
GROUP BY ym
ORDER BY ym
""")
conn.commit()

# Podgląd
df_master = pd.read_sql('SELECT * FROM v_master LIMIT 5', conn)
print('v_master - pierwsze 5 wierszy:')
print(df_master.to_string(index=False))

v_master - pierwsze 5 wierszy:
     ym  fedfunds   m2sl      sp500    t10y2y  usrec  unrate     gdpc1  cpiaucsl     icsa   payems  indpro rsxfs  permit hy_spread  credit_cond  lei       vix
1990-01      8.23 3166.3 339.971818  0.121429    0.0     5.4 10047.386     127.5 361000.0 109196.0 61.7290  None  1748.0      None          NaN 1.71 23.347273
1990-02      8.24 3178.2 330.452635  0.102632    0.0     5.3       NaN     128.0 358250.0 109436.0 62.2896  None  1329.0      None          NaN 1.45 23.262632
1990-03      8.28 3189.2 338.465000 -0.038182    0.0     5.2       NaN     128.6 345200.0 109640.0 62.5999  None  1246.0      None          NaN 1.67 20.062273
1990-04      8.26 3201.2 338.177998  0.061500    0.0     5.4 10083.855     128.9 361750.0 109674.0 62.4359  None  1136.0      None         54.4 1.31 21.403500
1990-05      8.18 3201.3 350.250001  0.115909    0.0     5.4       NaN     129.1 355250.0 109828.0 62.6258  None  1067.0      None          NaN 1.10 18.097727


---
# 10 Zapytań Analitycznych

Każde zapytanie odpowiada na konkretne pytanie biznesowe.

In [5]:
def sql(query, opis=''):
    """Uruchom zapytanie i wyświetl wynik z opisem."""
    if opis:
        print(f'\n{'='*60}')
        print(f'  {opis}')
        print('='*60)
    df = pd.read_sql(query, conn)
    print(df.to_string(index=False))
    return df

In [6]:
# ZAPYTANIE 1
# Kiedy yield curve była odwrócona (T10Y2Y < 0)?
# Sygnał ostrzegawczy przed recesją

q1 = sql("""
SELECT
    ym,
    ROUND(t10y2y, 3) AS yield_spread,
    ROUND(fedfunds, 2) AS stopa_fed,
    CASE WHEN usrec = 1 THEN 'RECESJA' ELSE '' END AS recesja
FROM v_master
WHERE t10y2y < 0
ORDER BY ym
""", 'Q1: Okresy odwróconej krzywej dochodowości (yield curve inversion)')


  Q1: Okresy odwróconej krzywej dochodowości (yield curve inversion)
     ym  yield_spread  stopa_fed recesja
1990-03        -0.038       8.28        
1998-06        -0.024       5.56        
2000-02        -0.091       5.73        
2000-03        -0.272       5.85        
2000-04        -0.413       6.02        
2000-05        -0.369       6.27        
2000-06        -0.385       6.53        
2000-07        -0.285       6.54        
2000-08        -0.403       6.50        
2000-09        -0.283       6.52        
2000-10        -0.174       6.51        
2000-11        -0.158       6.51        
2000-12        -0.112       6.40        
2006-02        -0.099       4.49        
2006-03        -0.010       4.59        
2006-06        -0.015       4.99        
2006-07        -0.031       5.24        
2006-08        -0.027       5.25        
2006-09        -0.050       5.25        
2006-10        -0.067       5.25        
2006-11        -0.145       5.25        
2006-12        -0.109       

In [7]:
# ZAPYTANIE 2
# Wszystkie epizody odwroconej krzywej i to, czy recesja przyszla w ciagu 36 miesiecy.
# Epizod = ciag kolejnych miesiecy z t10y2y < 0, wyznaczony technika gaps and islands
# (roznica dwoch numeracji jest stala w obrebie jednego ciagu).
# UWAGA: ta wersja NIE scala epizodow oddzielonych krotka przerwa - np. 2006-02, 2006-06
# i 2007-05 to tu trzy epizody, a notebook 05 traktuje je jako jedna inwersje przed GFC.
# Scalanie jest trywialne w Pythonie i tam wlasnie siedzi; SQL odpowiada na pytanie
# prostsze, ale odpowiada na nie poprawnie.

q2 = sql("""
WITH ponumerowane AS (
    SELECT ym, t10y2y, ROW_NUMBER() OVER (ORDER BY ym) AS nr
    FROM v_master
    WHERE t10y2y IS NOT NULL
),
ujemne AS (
    SELECT ym, nr - ROW_NUMBER() OVER (ORDER BY ym) AS grupa
    FROM ponumerowane
    WHERE t10y2y < 0
),
epizody AS (
    SELECT MIN(ym) AS start_inwersji, MAX(ym) AS koniec_inwersji, COUNT(*) AS dlugosc_mies
    FROM ujemne
    GROUP BY grupa
),
starty_recesji AS (
    SELECT ym FROM (
        SELECT ym, usrec, LAG(usrec) OVER (ORDER BY ym) AS poprzedni
        FROM v_master
        WHERE usrec IS NOT NULL
    )
    WHERE usrec = 1 AND (poprzedni = 0 OR poprzedni IS NULL)
)
SELECT
    e.start_inwersji,
    e.koniec_inwersji,
    e.dlugosc_mies,
    MIN(r.ym) AS recesja_po,
    CAST((julianday(MIN(r.ym) || '-01') - julianday(e.start_inwersji || '-01')) / 30.44 AS INT)
        AS miesiecy_do_recesji
FROM epizody e
LEFT JOIN starty_recesji r
    ON r.ym > e.start_inwersji
    AND (julianday(r.ym || '-01') - julianday(e.start_inwersji || '-01')) / 30.44 <= 36
GROUP BY e.start_inwersji, e.koniec_inwersji, e.dlugosc_mies
ORDER BY e.start_inwersji
""", 'Q2: Epizody inwersji krzywej i recesja w ciagu 36 miesiecy')



  Q2: Epizody inwersji krzywej i recesja w ciagu 36 miesiecy
start_inwersji koniec_inwersji  dlugosc_mies recesja_po  miesiecy_do_recesji
       1990-03         1990-03             1    1990-08                  5.0
       1998-06         1998-06             1    2001-04                 34.0
       2000-02         2000-12            11    2001-04                 13.0
       2006-02         2006-03             2    2008-01                 22.0
       2006-06         2007-03            10    2008-01                 19.0
       2007-05         2007-05             1    2008-01                  8.0
       2022-07         2024-08            26        NaN                  NaN


In [8]:
# ZAPYTANIE 3
# Średni wzrost M2 w 12 miesiącach PRZED każdą recesją vs poza recesją
# Czy dodruk pieniądza poprzedza recesje?

q3 = sql("""
WITH m2_growth AS (
    SELECT
        ym,
        m2sl,
        usrec,
        LAG(m2sl, 12) OVER (ORDER BY ym) AS m2sl_12m_ago,
        LEAD(usrec, 6) OVER (ORDER BY ym) AS recession_in_6m
    FROM v_master
    WHERE m2sl IS NOT NULL AND usrec IS NOT NULL
)
SELECT
    CASE
        WHEN recession_in_6m = 1 THEN 'Przed recesją (6m wcześniej)'
        WHEN usrec = 1           THEN 'Podczas recesji'
        ELSE                          'Normalny wzrost'
    END AS okres,
    COUNT(*) AS n_miesiecy,
    ROUND(AVG((m2sl - m2sl_12m_ago) / m2sl_12m_ago * 100), 2) AS sredni_wzrost_M2_pct
FROM m2_growth
WHERE m2sl_12m_ago IS NOT NULL AND recession_in_6m IS NOT NULL
GROUP BY 1
ORDER BY sredni_wzrost_M2_pct DESC
""", 'Q3: Średni wzrost M2 YoY - przed recesją vs normalny okres')


  Q3: Średni wzrost M2 YoY - przed recesją vs normalny okres
                       okres  n_miesiecy  sredni_wzrost_M2_pct
             Podczas recesji          17                  8.93
Przed recesją (6m wcześniej)          28                  6.65
             Normalny wzrost         376                  5.50


In [9]:
# ZAPYTANIE 4
# Zasieg spadku S&P500 WEWNATRZ okna recesji: najwyzszy i najnizszy odczyt
# miesieczny miedzy szczytem a dolkiem wg NBER. To nie jest max drawdown liczony
# od szczytu poprzedzajacego recesje - takie okno bylo by szersze niz dane NBER.

q4 = sql("""
WITH rec_periods AS (
    SELECT name, start, end FROM dim_recession
),
sp_during AS (
    SELECT
        r.name,
        r.start,
        r.end,
        MAX(m.sp500) AS sp500_peak,
        MIN(m.sp500) AS sp500_trough
    FROM rec_periods r
    JOIN v_master m
        ON m.ym BETWEEN strftime('%Y-%m', r.start) AND strftime('%Y-%m', r.end)
    WHERE m.sp500 IS NOT NULL
    GROUP BY r.name, r.start, r.end
)
SELECT
    name AS recesja,
    start || ' → ' || end AS okres,
    ROUND(sp500_peak, 0)  AS sp500_szczyt,
    ROUND(sp500_trough, 0) AS sp500_dolek,
    ROUND((sp500_trough - sp500_peak) / sp500_peak * 100, 1) AS drawdown_pct
FROM sp_during
ORDER BY start
""", 'Q4: Max drawdown S&P500 podczas każdej recesji')


  Q4: Max drawdown S&P500 podczas każdej recesji


 recesja                   okres  sp500_szczyt  sp500_dolek  drawdown_pct
Gulf War 1990-07-01 → 1991-03-01         372.0        307.0         -17.5
 Dot-com 2001-03-01 → 2001-11-01        1270.0       1045.0         -17.8
     GFC 2007-12-01 → 2009-06-01        1479.0        757.0         -48.8
   COVID 2020-02-01 → 2020-04-01        3277.0       2652.0         -19.1

In [10]:
# ZAPYTANIE 5
# Stopa Fed i CPI - realna stopa procentowa per dekada
# Realna stopa = FEDFUNDS - CPI_YoY

q5 = sql("""
WITH cpi_growth AS (
    SELECT
        ym,
        fedfunds,
        cpiaucsl,
        LAG(cpiaucsl, 12) OVER (ORDER BY ym) AS cpi_12m_ago
    FROM v_master
    WHERE fedfunds IS NOT NULL AND cpiaucsl IS NOT NULL
)
SELECT
    substr(ym, 1, 3) || '0s' AS dekada,
    ROUND(AVG(fedfunds), 2)  AS avg_stopa_fed,
    ROUND(AVG((cpiaucsl - cpi_12m_ago) / cpi_12m_ago * 100), 2) AS avg_inflacja_pct,
    ROUND(AVG(fedfunds - (cpiaucsl - cpi_12m_ago) / cpi_12m_ago * 100), 2) AS avg_realna_stopa
FROM cpi_growth
WHERE cpi_12m_ago IS NOT NULL
GROUP BY dekada
ORDER BY dekada
""", 'Q5: Średnia realna stopa procentowa per dekada (Fed minus inflacja)')


  Q5: Średnia realna stopa procentowa per dekada (Fed minus inflacja)
dekada  avg_stopa_fed  avg_inflacja_pct  avg_realna_stopa
 1990s           4.82              2.74              2.08
 2000s           2.96              2.57              0.38
 2010s           0.61              1.77             -1.16
 2020s           2.83              3.93             -1.11


In [11]:
# ZAPYTANIE 6
# Wzrost S&P500 w roku po obniżce stóp Fed
# Czy rynek reaguje wzrostem gdy Fed luzuje politykę?

q6 = sql("""
WITH fed_changes AS (
    SELECT
        ym,
        fedfunds,
        LAG(fedfunds) OVER (ORDER BY ym) AS prev_fedfunds,
        sp500,
        LEAD(sp500, 12) OVER (ORDER BY ym) AS sp500_12m_later
    FROM v_master
    WHERE fedfunds IS NOT NULL AND sp500 IS NOT NULL
)
SELECT
    CASE
        WHEN fedfunds < prev_fedfunds THEN 'Obniżka stóp'
        WHEN fedfunds > prev_fedfunds THEN 'Podwyżka stóp'
        ELSE                               'Brak zmiany'
    END AS decyzja_fed,
    COUNT(*) AS n_miesiecy,
    ROUND(AVG((sp500_12m_later - sp500) / sp500 * 100), 1) AS avg_zwrot_sp500_12m_pct
FROM fed_changes
WHERE prev_fedfunds IS NOT NULL AND sp500_12m_later IS NOT NULL
GROUP BY 1
ORDER BY avg_zwrot_sp500_12m_pct DESC
""", 'Q6: Średni zwrot S&P500 w 12 miesiącach po decyzji Fed')


  Q6: Średni zwrot S&P500 w 12 miesiącach po decyzji Fed
  decyzja_fed  n_miesiecy  avg_zwrot_sp500_12m_pct
Podwyżka stóp         190                     11.7
  Brak zmiany          82                     11.2
 Obniżka stóp         155                      7.7


In [12]:
# ZAPYTANIE 7
# Porównanie warunków makro 12 miesięcy przed każdą recesją
# Tablica pomocna do Recession Scorecard
#
# M2 YoY liczone w osobnym CTE na calym v_master. Wczesniej LAG(m2sl, 12) stal
# w zewnetrznym SELECT, czyli liczyl sie PO joinie na trzech wierszach - nie mial
# skad wziac dwunastu wierszy wstecz i kolumna wychodzila pusta w kazdym wierszu.
#
# Gulf War (1990-07) nie ma tu wiersza: punkt pomiaru wypada w 1989-07, przed
# poczatkiem danych. To ograniczenie zakresu, nie blad zapytania.

q7 = sql("""
WITH m2_yoy AS (
    SELECT
        ym,
        (m2sl - LAG(m2sl, 12) OVER (ORDER BY ym)) / LAG(m2sl, 12) OVER (ORDER BY ym) * 100 AS m2_yoy_pct
    FROM v_master
),
pre_rec AS (
    SELECT
        r.name,
        strftime('%Y-%m', date(r.start, '-12 months')) AS ym_12m_before
    FROM dim_recession r
)
SELECT
    p.name AS recesja,
    p.ym_12m_before AS data_pomiaru,
    ROUND(m.fedfunds, 2) AS stopa_fed,
    ROUND(m.t10y2y, 3)  AS yield_curve,
    ROUND(m.unrate, 1)  AS bezrobocie,
    ROUND(g.m2_yoy_pct, 1) AS m2_yoy_pct
FROM pre_rec p
JOIN v_master m ON m.ym = p.ym_12m_before
JOIN m2_yoy   g ON g.ym = p.ym_12m_before
ORDER BY p.ym_12m_before
""", 'Q7: Warunki makro 12 miesięcy przed każdą recesją')



  Q7: Warunki makro 12 miesięcy przed każdą recesją


recesja data_pomiaru  stopa_fed  yield_curve  bezrobocie  m2_yoy_pct
Dot-com      2000-03       5.85       -0.272         4.0         6.3
    GFC      2006-12       5.24       -0.109         4.4         5.9
  COVID      2019-02       2.40        0.172         3.8         4.0

In [13]:
# ZAPYTANIE 8
# Bezrobocie - sygnał Sahm Rule
# Sahm Rule: recesja gdy bezrobocie rośnie o 0.5pp ponad minimum z ostatnich 12m

q8 = sql("""
WITH sahm AS (
    SELECT
        ym,
        unrate,
        usrec,
        MIN(unrate) OVER (ORDER BY ym ROWS BETWEEN 11 PRECEDING AND CURRENT ROW) AS min_12m,
        AVG(unrate) OVER (ORDER BY ym ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)  AS avg_3m
    FROM v_master
    WHERE unrate IS NOT NULL AND usrec IS NOT NULL
)
SELECT
    ym,
    ROUND(unrate, 1)             AS bezrobocie,
    ROUND(avg_3m - min_12m, 2)  AS sahm_indicator,
    CASE WHEN avg_3m - min_12m >= 0.5 THEN 'SYGNAŁ' ELSE '' END AS sahm_signal,
    CASE WHEN usrec = 1 THEN 'RECESJA' ELSE '' END AS recesja
FROM sahm
WHERE avg_3m - min_12m >= 0.4  -- pokaż tylko miesiące bliskie sygnałowi
ORDER BY ym
""", 'Q8: Sahm Rule - sygnały ostrzegawcze bezrobocia')


  Q8: Sahm Rule - sygnały ostrzegawcze bezrobocia


     ym  bezrobocie  sahm_indicator sahm_signal recesja
1990-09         5.9            0.50      SYGNAŁ RECESJA
1990-10         5.9            0.63      SYGNAŁ RECESJA
1990-11         6.2            0.80      SYGNAŁ RECESJA
1990-12         6.3            0.93      SYGNAŁ RECESJA
1991-01         6.4            1.10      SYGNAŁ RECESJA
1991-02         6.6            1.23      SYGNAŁ RECESJA
1991-03         6.8            1.40      SYGNAŁ RECESJA
1991-04         6.7            1.50      SYGNAŁ        
1991-05         6.9            1.60      SYGNAŁ        
1991-06         6.9            1.33      SYGNAŁ        
1991-07         6.8            1.17      SYGNAŁ        
1991-08         6.9            0.97      SYGNAŁ        
1991-09         6.9            0.97      SYGNAŁ        
1991-10         7.0            0.73      SYGNAŁ        
1991-11         7.0            0.67      SYGNAŁ        
1991-12         7.3            0.70      SYGNAŁ        
1992-01         7.3            0.60      SYGNAŁ 

In [14]:
# ZAPYTANIE 9
# Czas trwania i głębokość każdej recesji

q9 = sql("""
SELECT
    r.name AS recesja,
    r.start,
    r.end,
    CAST((julianday(r.end) - julianday(r.start)) / 30 AS INT) AS dlugosc_miesiecy,
    r.cause AS przyczyna
FROM dim_recession r
ORDER BY r.start
""", 'Q9: Czas trwania każdej recesji')


  Q9: Czas trwania każdej recesji
 recesja      start        end  dlugosc_miesiecy                                            przyczyna
Gulf War 1990-07-01 1991-03-01                 8 Szok naftowy po inwazji na Kuwejt i zacieśnienie Fed
 Dot-com 2001-03-01 2001-11-01                 8                      Pęknięcie bańki technologicznej
     GFC 2007-12-01 2009-06-01                18               Kryzys finansowy - rynek nieruchomości
   COVID 2020-02-01 2020-04-01                 2                                    Pandemia COVID-19


In [15]:
# ZAPYTANIE 10
# AKTUALNY STAN (ostatnie 6 miesięcy) - wejście do Recession Scorecard

q10 = sql("""
WITH recent AS (
    SELECT *
    FROM v_master
    WHERE fedfunds IS NOT NULL
    ORDER BY ym DESC
    LIMIT 6
),
m2_calc AS (
    SELECT
        r.ym,
        r.fedfunds,
        r.t10y2y,
        r.unrate,
        r.m2sl,
        r.cpiaucsl,
        r.sp500,
        h.m2sl AS m2sl_12m_ago,
        h.cpiaucsl AS cpi_12m_ago
    FROM recent r
    LEFT JOIN v_master h
        ON h.ym = strftime('%Y-%m', date(r.ym || '-01', '-12 months'))
)
SELECT
    ym AS miesiac,
    ROUND(fedfunds, 2)                                           AS stopa_fed,
    ROUND(t10y2y, 3)                                            AS yield_curve,
    ROUND(unrate, 1)                                            AS bezrobocie,
    ROUND((m2sl - m2sl_12m_ago) / m2sl_12m_ago * 100, 1)      AS m2_yoy_pct,
    ROUND(fedfunds - (cpiaucsl - cpi_12m_ago)
          / cpi_12m_ago * 100, 2)                               AS realna_stopa,
    ROUND(sp500, 0)                                             AS sp500,
    CASE WHEN t10y2y < 0 THEN '⚠️' ELSE '✅' END               AS yield_ok,
    CASE WHEN (m2sl - m2sl_12m_ago) / m2sl_12m_ago * 100 < 0
         THEN '⚠️' ELSE '✅' END                                AS m2_ok
FROM m2_calc
WHERE m2sl_12m_ago IS NOT NULL
ORDER BY miesiac DESC
""", 'Q10: Aktualny stan wskaźników makro (ostatnie 6 miesięcy)')


  Q10: Aktualny stan wskaźników makro (ostatnie 6 miesięcy)
miesiac  stopa_fed  yield_curve  bezrobocie  m2_yoy_pct  realna_stopa  sp500 yield_ok m2_ok
2026-08       3.63        0.468         4.1         NaN          0.28 7711.0        ✅     ✅
2026-07       3.63        0.377         4.1         5.4          0.33 7481.0        ✅     ✅
2026-06       3.63        0.358         4.2         5.3          0.17 7450.0        ✅     ✅
2026-05       3.63        0.489         4.3         5.4         -0.54 7413.0        ✅     ✅
2026-04       3.64        0.520         4.3         4.5         -0.14 6957.0        ✅     ✅
2026-03       3.64        0.531         4.3         4.4          0.35 6654.0        ✅     ✅


## Podsumowanie - co mamy w bazie

In [16]:
print('STRUKTURA BAZY:', DB_PATH.name)
print()

tabele = pd.read_sql(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY type, name",
    conn
)
for _, row in tabele.iterrows():
    count_query = f"SELECT COUNT(*) as n FROM {row['name']}"
    try:
        n = pd.read_sql(count_query, conn).iloc[0, 0]
        print(f"  [{row['type']:5}] {row['name']:<20} {n:>6} wierszy")
    except:
        print(f"  [{row['type']:5}] {row['name']:<20}")

conn.close()
print(f'\nBaza gotowa do Dnia 3 (wskaźniki pochodne w Pythonie).')

STRUKTURA BAZY: fed_cycles.db

  [table] dim_recession             4 wierszy
  [table] raw_series            24315 wierszy


  [view ] v_master                441 wierszy

Baza gotowa do Dnia 3 (wskaźniki pochodne w Pythonie).
